# Intelligent Customer Service Agent — Demo
## LLM Project 1 | ReAct + LangGraph

Demonstrates all **11 required functions** from Section 9 of the project specification.

### LangGraph Workflow
```
User Input
    |
[Planner Node]   ← ReAct: extract intent + entities, select tools
    |
[Tool Node(s)]   ← MySQL queries, business logic
    |
[Verifier Node]  ← prevent hallucinations, enforce policy
    |
Final Response
```

### Test Score Checklist

| # | Function | Test Query | Expected Behavior |
|---|----------|-----------|-------------------|
| 1 | Intent Parsing | Where is my order 12345? | Extract intent=tracking, order_id=12345 |
| 2 | OrderLookupTool | Check status of order 1001 | MySQL SELECT from orders |
| 3 | CustomerProfileTool | Show my profile | MySQL SELECT from customers |
| 4 | RefundTool | Refund order 5678 | MySQL UPDATE status=refund_requested |
| 5 | ComplaintLoggerTool | Complain about order 2222 | MySQL INSERT into complaints |
| 6 | Multi-step Reasoning | Refund 7890 if delivered | order_lookup → conditional refund |
| 7 | Short-Term Memory | Cancel it (after prior query) | recall order_id from session history |
| 8 | LTM Read | What issues have I had before? | SELECT customer_memory |
| 9 | LTM Write | Remember I prefer refunds | INSERT customer_memory |
| 10 | Personalization | My order is late again | detect repeated issue from LTM |
| 11 | Verifier | Refund order 0000 | reject — order not found |

---
## Setup — Load Agent

In [ ]:
import sys, os, uuid, warnings
warnings.filterwarnings("ignore")

sys.path.insert(0, os.getcwd())

from main import app
from langchain_core.messages import HumanMessage, AIMessage

print("Agent loaded successfully.")
print("STM  : LangGraph MemorySaver (per thread_id)")
print("LTM  : MySQL  customer_memory  table")
print("Tools: order_lookup, customer_profile, request_refund,")
print("       log_complaint, read_long_term_memory, write_long_term_memory")

In [ ]:
def run_query(user_input: str, customer_id: int, thread_id: str = None) -> tuple:
    """Run a query through the ReAct agent and print the full execution trace."""
    if thread_id is None:
        thread_id = f"s_{uuid.uuid4().hex[:6]}"
    cfg = {"configurable": {"thread_id": thread_id, "customer_id": customer_id}}

    sep = "=" * 72
    print(f"\n{sep}")
    print(f"  QUERY   : {user_input}")
    print(f"  Customer: {customer_id}  |  Thread: {thread_id}")
    print(f"{sep}")

    events = app.stream(
        {"messages": [HumanMessage(content=user_input)]},
        cfg,
        stream_mode="values",
    )

    final_response = None
    for event in events:
        msg = event["messages"][-1]
        if isinstance(msg, AIMessage) and msg.tool_calls:
            for tc in msg.tool_calls:
                print(f"\n  [Planner] selected tool : {tc['name']}")
                print(f"            arguments    : {tc.get('args', {})}")
        elif msg.type == "tool":
            preview = msg.content[:400] + ("..." if len(msg.content) > 400 else "")
            print(f"  [Tool: {msg.name}]")
            print(f"    {preview}")
        elif isinstance(msg, AIMessage) and not msg.tool_calls:
            final_response = msg.content

    print(f"\n  [Final Response]")
    print(f"  {'-' * 60}")
    for line in (final_response or "(no response)").split("\n"):
        print(f"  {line}")
    print()
    return final_response, cfg

print("run_query() ready.")

---
## Test 1 — Intent Parsing

**Query**: `Where is my order 12345?`  
**Expected**: Planner extracts `intent=tracking`, `order_id=12345` and calls `order_lookup`  
**Data**: Alice (customer_id=1) owns order 12345 — Wireless Mouse, status: **shipped**

In [ ]:
run_query("Where is my order 12345?", customer_id=1)

---
## Test 2 — OrderLookupTool

**Query**: `Check status of order 1001`  
**Expected**: `order_lookup` executes `SELECT * FROM orders WHERE order_id=1001`  
**Data**: Bob (customer_id=2) owns order 1001 — Mechanical Keyboard, status: **processing**

In [ ]:
run_query("Check status of order 1001", customer_id=2)

---
## Test 3 — CustomerProfileTool

**Query**: `Show my profile`  
**Expected**: `customer_profile` executes `SELECT * FROM customers WHERE customer_id=?`  
**Data**: Alice (customer_id=1) — alice@example.com

In [ ]:
run_query("Show my profile", customer_id=1)

---
## Test 4 — RefundTool

**Query**: `Refund order 5678`  
**Expected**: `request_refund` runs `UPDATE orders SET status='refund_requested' WHERE order_id=5678`  
**Data**: Alice (customer_id=1) owns order 5678 — Headphones, status: **delivered** → eligible for refund

In [ ]:
run_query("Refund order 5678", customer_id=1)

---
## Test 5 — ComplaintLoggerTool

**Query**: `I want to complain about order 2222`  
**Expected**: `log_complaint` runs `INSERT INTO complaints (...)`  
**Data**: Charlie (customer_id=3) owns order 2222 — Ergonomic Chair, status: **delivered**

In [ ]:
run_query("I want to complain about order 2222, the item arrived damaged", customer_id=3)

---
## Test 6 — Multi-step Reasoning

**Query**: `Refund order 7890 only if it has already been delivered`  
**Expected**: Planner chains two tool calls — `order_lookup` first to verify delivery, then `request_refund`  
**Data**: Bob (customer_id=2) owns order 7890 — USB-C Hub, status: **delivered** → refund proceeds

In [ ]:
run_query("Refund order 7890 only if it has already been delivered", customer_id=2)

---
## Test 7 — Short-Term Memory (STM)

**Expected**: Two queries sent in the **same `thread_id`**. The second query resolves `"it"` from the prior turn's context stored in `MemorySaver`.

> The same `thread_id` keeps the full message history across `run_query` calls.  
> On Turn 2, the agent sees both messages and infers `order_id=1001` without being told.

- **Turn 1**: `What is the status of order 1001?` — establishes order context  
- **Turn 2**: `Cancel it` — agent uses STM to resolve `"it"` → order 1001

In [ ]:
STM_THREAD = "stm_demo_001"

print(">>> TURN 1: Establish order context")
run_query("What is the status of order 1001?", customer_id=2, thread_id=STM_THREAD)

In [ ]:
print(">>> TURN 2: Follow-up — agent resolves 'it' from STM (same thread)")
run_query("Cancel it", customer_id=2, thread_id=STM_THREAD)

---
## Test 8 — Long-Term Memory (Read)

**Query**: `What issues have I had before?`  
**Expected**: `read_long_term_memory` executes `SELECT key, value FROM customer_memory WHERE customer_id=3`  
**Data**: Charlie (customer_id=3) has pre-seeded record: `past_issues: frequent late deliveries`

In [ ]:
run_query("What issues have I had before?", customer_id=3)

---
## Test 9 — Long-Term Memory (Write)

**Query**: `Remember I prefer refunds over store credit`  
**Expected**: `write_long_term_memory` executes `INSERT INTO customer_memory (customer_id, key, value)`  
**Data**: Alice (customer_id=1)

In [ ]:
run_query("Remember I prefer refunds over store credit", customer_id=1)

---
## Test 10 — Personalization

**Query**: `My order is late again!`  
**Expected**: Agent calls `read_long_term_memory`, detects the **repeated pattern** (`frequent late deliveries`), and responds with elevated priority / apology  
**Data**: Charlie (customer_id=3) — LTM has `past_issues: frequent late deliveries`

In [ ]:
run_query("My order is late again!", customer_id=3)

---
## Test 11 — Verifier

**Query**: `Refund order 0000`  
**Expected**: `order_lookup` returns "not found" → Verifier rewrites the response to reject the request safely  
**Data**: Order 0000 does **not** exist in the database

In [ ]:
run_query("Refund order 0000", customer_id=1)

---
## Summary — All 11 Functions Covered

| # | Function | Tool / Mechanism | MySQL Operation |
|---|----------|-----------------|----------------|
| 1 | Intent Parsing | Planner (LLM reasoning) | — |
| 2 | OrderLookupTool | `order_lookup` | SELECT orders |
| 3 | CustomerProfileTool | `customer_profile` | SELECT customers |
| 4 | RefundTool | `request_refund` | UPDATE orders |
| 5 | ComplaintLoggerTool | `log_complaint` | INSERT complaints |
| 6 | Multi-step Reasoning | Planner → tool chaining | SELECT + UPDATE |
| 7 | Short-Term Memory | LangGraph MemorySaver | — (in-process) |
| 8 | LTM Read | `read_long_term_memory` | SELECT customer_memory |
| 9 | LTM Write | `write_long_term_memory` | INSERT customer_memory |
| 10 | Personalization | LTM read + Planner reasoning | SELECT customer_memory |
| 11 | Verifier | Verifier Node (LLM) | — |

> **Demo tip**: Re-run any cell to repeat the demo live.  
> Test 7 requires running Turn 1 before Turn 2 to populate the STM.